In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import accuracy_score
import joblib

print("Building the Master Ensemble Voting Model...")

# 1. Load Data
matches = pd.read_csv('../data/raw/matches.csv')
venue_intel = pd.read_csv('../data/processed/venue_intelligence.csv')

# Clean Team Names
team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad', 'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru', 'Gujarat Lions': 'Gujarat Titans'
}
matches.replace(team_mapping, inplace=True)
matches.dropna(subset=['winner', 'venue'], inplace=True)

# 2. Target Variable
matches['target'] = (matches['team1'] == matches['winner']).astype(int)

# 3. Basic Features (We will add the heavy engineered features in the Recency Weighting step)
venue_win_dict = dict(zip(venue_intel['venue'], venue_intel['bat_first_win_pct']))
matches['venue_bat_first_win_pct'] = matches['venue'].map(venue_win_dict).fillna(50.0)

ml_df = matches[['team1', 'team2', 'venue', 'toss_decision', 'venue_bat_first_win_pct', 'target']].copy()

# 4. Train-Test Split (Chronological)
X = ml_df.drop('target', axis=1)
y = ml_df['target']
split_idx = int(len(ml_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 5. The Preprocessing Step
preprocessor = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), 
     ['team1', 'team2', 'venue', 'toss_decision'])
], remainder='passthrough')

# 6. Define the Base Models
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_xgb = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, use_label_encoder=False, eval_metric='logloss', random_state=42)

# 7. Create the Ensemble Voting Classifier
# voting='soft' means it averages the probability outputs (crucial for our Streamlit UI!)
ensemble = VotingClassifier(
    estimators=[
        ('lr', model_lr),
        ('rf', model_rf),
        ('xgb', model_xgb)
    ],
    voting='soft'
)

# 8. Build and Train the Master Pipeline
master_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('ensemble', ensemble)
])

print("Training Ensemble (Logistic Regression + Random Forest + XGBoost)...")
master_pipeline.fit(X_train, y_train)

# 9. Evaluate
y_pred = master_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🌟 Master Ensemble Pre-Match Accuracy: {accuracy * 100:.2f}%")

# 10. Save the Master Model
os.makedirs('../models', exist_ok=True)
joblib.dump(master_pipeline, '../models/ensemble_pre_match.pkl')
joblib.dump(list(X.columns), '../models/ensemble_features.pkl')

print("✅ Master Ensemble model saved to 'models/ensemble_pre_match.pkl'")

Building the Master Ensemble Voting Model...
Training Ensemble (Logistic Regression + Random Forest + XGBoost)...


c:\Users\garvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:41:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\garvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


🌟 Master Ensemble Pre-Match Accuracy: 47.71%
✅ Master Ensemble model saved to 'models/ensemble_pre_match.pkl'
